# 🦀 Day 6: Ownership in Functions

Today:
- Passing ownership vs borrowing in function parameters
- Return value ownership
- Common patterns and best practices

---

## 🔀 Function Parameter Patterns

In [ ]:
// Pattern 1: Take ownership (fn consumes the value)
fn consume(s: String) {
    println!("Consumed: {}", s);
} // s is dropped here

// Pattern 2: Borrow immutably (fn reads but doesn't modify)
fn inspect(s: &str) {
    println!("Inspected: {} (len={})", s, s.len());
}

// Pattern 3: Borrow mutably (fn modifies in place)
fn append(s: &mut String, suffix: &str) {
    s.push_str(suffix);
}

// Pattern 4: Take ownership and return (transform)
fn to_uppercase_owned(s: String) -> String {
    s.to_uppercase()
}

let mut name = String::from("alice");
inspect(&name);                    // Borrow — name still valid
append(&mut name, " smith");       // Mut borrow — modifies in place
println!("After append: {}", name);

let upper = to_uppercase_owned(name); // name is MOVED
// println!("{}", name);             // ❌ name was consumed
println!("Upper: {}", upper);

## 📋 When to Use Each Pattern

| Pattern | Use When |
|---------|----------|
| `fn(T)` — take ownership | Function needs to keep/store the value, or is a constructor/transformer |
| `fn(&T)` — immutable borrow | Function only needs to read |
| `fn(&mut T)` — mutable borrow | Function needs to modify the value in place |
| `fn(T) -> T` — take and return | Transformation pipeline |

In [ ]:
// Real-world example: a struct that owns data
struct UserList {
    users: Vec<String>,
}

impl UserList {
    fn new() -> Self {
        UserList { users: Vec::new() }
    }

    // Takes ownership — the name is now owned by the struct
    fn add_user(&mut self, name: String) {
        self.users.push(name);
    }

    // Borrows — just reads
    fn contains(&self, name: &str) -> bool {
        self.users.iter().any(|u| u == name)
    }

    // Returns a slice — borrows from the struct
    fn all_users(&self) -> &[String] {
        &self.users
    }

    // Takes ownership of self — consumes the struct
    fn into_vec(self) -> Vec<String> {
        self.users
    }
}

let mut list = UserList::new();
list.add_user(String::from("Alice"));
list.add_user(String::from("Bob"));

println!("Contains Alice: {}", list.contains("Alice"));
println!("All: {:?}", list.all_users());

let users = list.into_vec(); // list is consumed
println!("Extracted: {:?}", users);
// println!("{:?}", list.all_users()); // ❌ list was consumed

## 🔄 Return Value Ownership

In [ ]:
// Returning owned values — ownership moves to caller
fn create_greeting(name: &str) -> String {
    format!("Hello, {}!", name)  // New String created, returned to caller
}

// Returning references — must tie to input lifetime
fn longest<'a>(a: &'a str, b: &'a str) -> &'a str {
    if a.len() >= b.len() { a } else { b }
}

// Returning Option with references
fn find_first_over(scores: &[i32], threshold: i32) -> Option<&i32> {
    scores.iter().find(|&&s| s > threshold)
}

let greeting = create_greeting("Rust");
println!("{}", greeting);

let result = longest("hello", "world!");
println!("Longest: {}", result);

let scores = vec![65, 72, 88, 91, 45];
if let Some(score) = find_first_over(&scores, 80) {
    println!("First over 80: {}", score);
}

## 🧩 Method Receiver Patterns

In [ ]:
#[derive(Debug)]
struct TextBuffer {
    content: String,
}

impl TextBuffer {
    fn new(text: &str) -> Self {
        TextBuffer { content: text.to_string() }
    }

    // &self — read-only
    fn len(&self) -> usize {
        self.content.len()
    }

    // &mut self — modify in place
    fn append(&mut self, text: &str) {
        self.content.push_str(text);
    }

    // self — consumes, transforms
    fn into_upper(self) -> TextBuffer {
        TextBuffer { content: self.content.to_uppercase() }
    }

    // &self returning &str — lifetime tied to self
    fn as_str(&self) -> &str {
        &self.content
    }
}

let mut buf = TextBuffer::new("hello");
println!("Len: {}", buf.len());       // &self
buf.append(" world");                  // &mut self
println!("Content: {}", buf.as_str()); // &self → &str

let upper = buf.into_upper();          // self — buf consumed
println!("Upper: {:?}", upper);

## 🏋️ Exercises

In [ ]:
// Exercise 1: Choose the right parameter types for these functions:
// a) fn print_info(???) — prints a person's name (doesn't modify)
// b) fn set_name(???) — changes a person's name
// c) fn extract_name(???) — removes the name from the person, returns it

#[derive(Debug)]
struct PersonEx {
    name: String,
    age: u32,
}

// YOUR CODE HERE


In [ ]:
// Exercise 2: Implement a Stack with proper ownership patterns
// - push(item: T) — takes ownership
// - peek(&self) -> Option<&T> — borrows
// - pop(&mut self) -> Option<T> — gives back ownership

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **`&T`** when you only need to read
2. **`&mut T`** when you need to modify in place
3. **`T`** when the function needs to own/store/consume the value
4. **`T → T`** for transformations that consume and produce
5. **Prefer borrowing** — only take ownership when you need it

---
**Tomorrow:** Mini-project — Text Analyzer! 📊